# OdiaGPT: English → Odia Transformer (From Scratch)
**Dataset:** AI4Bharat Samanantar — 500k sentence pairs  
**Architecture:** Encoder-Decoder Transformer | 6 blocks | 8 heads | d_model=512  
**GPU:** RTX 3050 6GB  
**Key fix:** Memory cleanup after every epoch — prevents RAM crash on long runs

---

## How to use this notebook

| Step | What it does | Run once or every time? |
|------|-------------|------------------------|
| STEP 1 | Memory clear + install | Every time you open notebook |
| STEP 2 | Imports + GPU setup | Every time |
| STEP 3 | Load dataset | Once (or after restart) |
| STEP 4 | Train tokenizers | Once only |
| STEP 5 | Dataset + DataLoaders | Once |
| STEP 6 | Build model | Once |
| STEP 7 | Train | Run to train or resume |
| STEP 8 | Load & translate | After training |
| STEP 9 | Evaluate scores | After training |

## Folder structure
```
OdiaLLM/
├── OdiaGPT_500k_Clean.ipynb   ← this file
├── odiagpt/                   ← checkpoints: model_1.pt, model_2.pt ...
├── tokenizer_en/              ← English BPE tokenizer
└── tokenizer_or/              ← Odia BPE tokenizer
```

## STEP 1: Memory Clear + Install Dependencies
⚠️ **Always run this cell first before anything else every time you open the notebook.**

In [2]:
import os

for epoch in range(1, 21):
    path = f"./odiagpt/model_{epoch}.pt"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"✅ model_{epoch:02d}.pt  →  {size_mb:.1f} MB")
    else:
        print(f"❌ model_{epoch:02d}.pt  →  NOT FOUND")

✅ model_01.pt  →  344.8 MB
✅ model_02.pt  →  344.8 MB
✅ model_03.pt  →  344.8 MB
✅ model_04.pt  →  344.8 MB
✅ model_05.pt  →  344.8 MB
✅ model_06.pt  →  344.8 MB
✅ model_07.pt  →  344.8 MB
✅ model_08.pt  →  344.8 MB
✅ model_09.pt  →  344.8 MB
✅ model_10.pt  →  344.8 MB
✅ model_11.pt  →  344.8 MB
✅ model_12.pt  →  344.8 MB
✅ model_13.pt  →  344.8 MB
✅ model_14.pt  →  344.8 MB
✅ model_15.pt  →  344.8 MB
✅ model_16.pt  →  344.8 MB
✅ model_17.pt  →  344.8 MB
✅ model_18.pt  →  344.8 MB
✅ model_19.pt  →  344.8 MB
✅ model_20.pt  →  344.8 MB


In [5]:
import torch

for epoch in [10, 18, 19, 20]:
    path = f"./odiagpt/model_{epoch}.pt"
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    saved_epoch = ckpt.get("epoch", "NOT FOUND")
    print(f"model_{epoch:02d}.pt  saved epoch = {saved_epoch}")

model_10.pt  saved epoch = 10
model_18.pt  saved epoch = 18
model_19.pt  saved epoch = 19
model_20.pt  saved epoch = 20


In [7]:
import torch

for epoch in [10, 18, 19, 20]:
    path = f"./odiagpt/model_{epoch}.pt"
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    state = ckpt["model_state_dict"]
    # Check file size of first layer — different training = different values
    first_layer = state["src_embed.embedding.weight"]
    print(f"model_{epoch:02d}.pt  shape={first_layer.shape}  mean={first_layer.mean():.6f}  std={first_layer.std():.6f}")
    print()

model_10.pt  shape=torch.Size([30000, 512])  mean=-0.001131  std=0.038700

model_18.pt  shape=torch.Size([30000, 512])  mean=-0.001689  std=0.050800

model_19.pt  shape=torch.Size([30000, 512])  mean=-0.000879  std=0.024000

model_20.pt  shape=torch.Size([30000, 512])  mean=-0.000923  std=0.024521



In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())



True

2.5.1+cu118
True


True

In [4]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import torch

# Clear any leftover memory from previous runs
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Memory cleared. Ready to start.")

✅ Memory cleared. Ready to start.


In [7]:
# Install required packages (skips if already installed)
!pip install datasets tokenizers sacrebleu tqdm torch --quiet
print("✅ All packages ready.")

✅ All packages ready.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## STEP 2: Imports & GPU Setup

In [5]:
import os
import gc
import math
import random
import torch
import torch.nn as nn
import sacrebleu
from torch.utils.data import Dataset, DataLoader, random_split
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tqdm import tqdm

# Create folders if they don't exist
for folder in ["./odiagpt", "./tokenizer_en", "./tokenizer_or"]:
    os.makedirs(folder, exist_ok=True)

# Setup GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU found — training will be very slow on CPU")

Using device: cuda
GPU : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.4 GB


## STEP 3: Load Dataset

**TRAIN_SIZE = 500000** is the safe setting for RTX 3050 6GB RAM.  
- 500k = ~2.5 hours per epoch  
- Changing to 700k risks RAM crash (memory fragmentation after epoch 5)  
- Do NOT change unless you have 16GB+ RAM

In [6]:
print("Loading Samanantar English-Odia dataset from HuggingFace...")
raw_dataset = load_dataset("ai4bharat/samanantar", "or", split="train")
print(f"Total available: {len(raw_dataset):,} sentence pairs")
print("Sample:", raw_dataset[0])

# ─────────────────────────────────────────────────
# DATASET SIZE — safe for RTX 3050 6GB
# 500k train, 3000 val
# ─────────────────────────────────────────────────
TRAIN_SIZE = 500000
VAL_SIZE   = 3000

if len(raw_dataset) >= TRAIN_SIZE + VAL_SIZE:
    raw_train_dataset, rest = random_split(
        raw_dataset, [TRAIN_SIZE, len(raw_dataset) - TRAIN_SIZE]
    )
    raw_val_dataset, _ = random_split(
        rest, [VAL_SIZE, len(rest) - VAL_SIZE]
    )
else:
    train_size = int(0.9 * len(raw_dataset))
    val_size   = len(raw_dataset) - train_size
    raw_train_dataset, raw_val_dataset = random_split(
        raw_dataset, [train_size, val_size]
    )

print(f"\nTrain size     : {len(raw_train_dataset):,}")
print(f"Validation size: {len(raw_val_dataset):,}")

# Free unused memory immediately after splitting
gc.collect()
print("✅ Dataset loaded and split done.")

Loading Samanantar English-Odia dataset from HuggingFace...


Total available: 998,228 sentence pairs
Sample: {'idx': 0, 'src': 'The Congress, however, has also not announced its candidates so far.', 'tgt': 'ଅଥଚ ବଡ଼ଚଣାର କଂଗ୍ରେସ ପ୍ରାର୍ଥୀ ଆଜି ପର୍ଯ୍ୟନ୍ତ ଘୋଷଣା କରାଯାଇପାରି ନାହିଁ।'}

Train size     : 500,000
Validation size: 3,000
✅ Dataset loaded and split done.


## STEP 4: Train BPE Tokenizers
⚠️ **Run this only once.** If tokenizer files already exist in `tokenizer_en/` and `tokenizer_or/`, skip this step and go to STEP 5.

In [7]:
def get_ds_iterator(dataset, key):
    """Yield text from dataset. key='src'=English, 'tgt'=Odia"""
    for data in dataset:
        text = data[key].strip()
        if text:
            yield text

SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

# ── English Tokenizer ──────────────────────────────────────
print("Training English tokenizer on 500k sentences...")
tokenizer_en = Tokenizer(BPE(unk_token="[UNK]"))
trainer_en   = BpeTrainer(min_frequency=2, vocab_size=30000, special_tokens=SPECIAL_TOKENS)
tokenizer_en.pre_tokenizer = Whitespace()
tokenizer_en.train_from_iterator(get_ds_iterator(raw_train_dataset, "src"), trainer=trainer_en)
tokenizer_en.save("./tokenizer_en/tokenizer_en.json")
print(f"English vocab size: {tokenizer_en.get_vocab_size()}")

# ── Odia Tokenizer ────────────────────────────────────────
print("Training Odia tokenizer on 500k sentences...")
tokenizer_or = Tokenizer(BPE(unk_token="[UNK]"))
trainer_or   = BpeTrainer(min_frequency=2, vocab_size=30000, special_tokens=SPECIAL_TOKENS)
tokenizer_or.pre_tokenizer = Whitespace()
tokenizer_or.train_from_iterator(get_ds_iterator(raw_train_dataset, "tgt"), trainer=trainer_or)
tokenizer_or.save("./tokenizer_or/tokenizer_or.json")
print(f"Odia vocab size: {tokenizer_or.get_vocab_size()}")

# Reload from saved files to confirm
tokenizer_en = Tokenizer.from_file("./tokenizer_en/tokenizer_en.json")
tokenizer_or = Tokenizer.from_file("./tokenizer_or/tokenizer_or.json")
print("✅ Tokenizers trained, saved and reloaded.")

Training English tokenizer on 500k sentences...
English vocab size: 30000
Training Odia tokenizer on 500k sentences...
Odia vocab size: 30000
✅ Tokenizers trained, saved and reloaded.


### Load tokenizers from saved files (use this if tokenizers already exist)

In [8]:
# Run this cell if tokenizers are already trained (skip STEP 4 training above)
tokenizer_en = Tokenizer.from_file("./tokenizer_en/tokenizer_en.json")
tokenizer_or = Tokenizer.from_file("./tokenizer_or/tokenizer_or.json")
print(f"English vocab: {tokenizer_en.get_vocab_size()}")
print(f"Odia vocab   : {tokenizer_or.get_vocab_size()}")
print("✅ Tokenizers loaded from files.")

English vocab: 30000
Odia vocab   : 30000
✅ Tokenizers loaded from files.


## STEP 5: Dataset Class & DataLoaders

In [9]:
# Fixed at 160 — safe for RTX 3050 6GB VRAM
# No scanning needed, 160 covers 99%+ of Samanantar sentences
max_seq_len = 160
print(f"max_seq_len = {max_seq_len}")


def causal_mask(size):
    """Upper triangular mask — stops decoder from seeing future tokens"""
    mask = torch.triu(torch.ones(1, size, size), diagonal=1).type(torch.int)
    return mask == 0


class EncodeDataset(Dataset):
    def __init__(self, raw_dataset, max_seq_len):
        super().__init__()
        self.raw_dataset = raw_dataset
        self.max_seq_len = max_seq_len
        # Cache special token IDs once
        self.CLS_ID = tokenizer_or.token_to_id("[CLS]")
        self.SEP_ID = tokenizer_or.token_to_id("[SEP]")
        self.PAD_ID = tokenizer_or.token_to_id("[PAD]")

    def __len__(self):
        return len(self.raw_dataset)

    def __getitem__(self, idx):
        raw         = self.raw_dataset[idx]
        source_text = raw["src"]   # English
        target_text = raw["tgt"]   # Odia

        src_ids = tokenizer_en.encode(source_text).ids[:self.max_seq_len - 2]
        tgt_ids = tokenizer_or.encode(target_text).ids[:self.max_seq_len - 1]

        src_pad_len = self.max_seq_len - len(src_ids) - 2
        tgt_pad_len = self.max_seq_len - len(tgt_ids) - 1

        # [CLS] + src tokens + [SEP] + padding
        encoder_input = torch.tensor(
            [self.CLS_ID] + src_ids + [self.SEP_ID] + [self.PAD_ID] * src_pad_len,
            dtype=torch.int64
        )
        # [CLS] + tgt tokens + padding  (teacher forcing input)
        decoder_input = torch.tensor(
            [self.CLS_ID] + tgt_ids + [self.PAD_ID] * tgt_pad_len,
            dtype=torch.int64
        )
        # tgt tokens + [SEP] + padding  (what model must predict)
        target_label = torch.tensor(
            tgt_ids + [self.SEP_ID] + [self.PAD_ID] * tgt_pad_len,
            dtype=torch.int64
        )

        encoder_mask = (encoder_input != self.PAD_ID).unsqueeze(0).unsqueeze(0).int()
        decoder_mask = (
            (decoder_input != self.PAD_ID).unsqueeze(0).unsqueeze(0).int()
            & causal_mask(self.max_seq_len)
        )

        return {
            "encoder_input": encoder_input,
            "decoder_input": decoder_input,
            "target_label":  target_label,
            "encoder_mask":  encoder_mask,
            "decoder_mask":  decoder_mask,
            "source_text":   source_text,
            "target_text":   target_text,
        }


train_ds = EncodeDataset(raw_train_dataset, max_seq_len)
val_ds   = EncodeDataset(raw_val_dataset,   max_seq_len)

# num_workers=0 is REQUIRED on Windows — do not change
train_dataloader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=0)
val_dataloader   = DataLoader(val_ds,   batch_size=1,  shuffle=False, num_workers=0)

print(f"Train batches : {len(train_dataloader):,}")
print(f"Val batches   : {len(val_dataloader):,}")
print("✅ DataLoaders ready.")

max_seq_len = 160
Train batches : 31,250
Val batches   : 3,000
✅ DataLoaders ready.


## STEP 6: Transformer Architecture (Built From Scratch)

In [10]:
# ══════════════════════════════════════════════════
# Embedding + Positional Encoding
# ══════════════════════════════════════════════════

class EmbeddingLayer(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model   = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int, dropout_rate: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        pe  = torch.zeros(max_seq_len, d_model)
        pos = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.shape[1], :].requires_grad_(False)
        return self.dropout(x)


# ══════════════════════════════════════════════════
# Multi-Head Attention
# ══════════════════════════════════════════════════

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout_rate: float):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.dropout   = nn.Dropout(dropout_rate)
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, q, k, v, mask):
        query = self.W_q(q)
        key   = self.W_k(k)
        value = self.W_v(v)
        B     = query.shape[0]
        query = query.view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        key   = key.view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        value = value.view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        scores = scores.softmax(dim=-1)
        scores = self.dropout(scores)
        out = scores @ value
        out = out.transpose(1, 2).contiguous().view(B, -1, self.num_heads * self.d_k)
        return self.W_o(out)


# ══════════════════════════════════════════════════
# FeedForward, LayerNorm, AddAndNorm
# ══════════════════════════════════════════════════

class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout_rate: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.layer_1 = nn.Linear(d_model, d_ff)
        self.layer_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.layer_2(self.dropout(torch.relu(self.layer_1(x))))


class LayerNorm(nn.Module):
    def __init__(self, d_model: int = 512, eps: float = 1e-5):
        super().__init__()
        self.eps   = eps
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta  = nn.Parameter(torch.zeros(d_model))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std  = x.std(dim=-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta


class AddAndNorm(nn.Module):
    def __init__(self, d_model: int, dropout_rate: float):
        super().__init__()
        self.dropout    = nn.Dropout(dropout_rate)
        self.layer_norm = LayerNorm(d_model)

    def forward(self, x, sub_layer):
        return x + self.dropout(sub_layer(self.layer_norm(x)))


# ══════════════════════════════════════════════════
# Encoder
# ══════════════════════════════════════════════════

class EncoderBlock(nn.Module):
    def __init__(self, d_model, multihead_attention, feed_forward, dropout_rate):
        super().__init__()
        self.multihead_attention = multihead_attention
        self.feed_forward        = feed_forward
        self.addnorm_1 = AddAndNorm(d_model, dropout_rate)
        self.addnorm_2 = AddAndNorm(d_model, dropout_rate)

    def forward(self, x, mask):
        x = self.addnorm_1(x, lambda x: self.multihead_attention(x, x, x, mask))
        x = self.addnorm_2(x, self.feed_forward)
        return x


class Encoder(nn.Module):
    def __init__(self, d_model, block_list: nn.ModuleList):
        super().__init__()
        self.block_list = block_list
        self.layer_norm = LayerNorm(d_model)

    def forward(self, x, mask):
        for block in self.block_list:
            x = block(x, mask)
        return self.layer_norm(x)


# ══════════════════════════════════════════════════
# Decoder
# ══════════════════════════════════════════════════

class DecoderBlock(nn.Module):
    def __init__(self, d_model, masked_attn, cross_attn, feed_forward, dropout_rate):
        super().__init__()
        self.masked_attn  = masked_attn
        self.cross_attn   = cross_attn
        self.feed_forward = feed_forward
        self.addnorm_1 = AddAndNorm(d_model, dropout_rate)
        self.addnorm_2 = AddAndNorm(d_model, dropout_rate)
        self.addnorm_3 = AddAndNorm(d_model, dropout_rate)

    def forward(self, x, enc_out, enc_mask, dec_mask):
        x = self.addnorm_1(x, lambda x: self.masked_attn(x, x, x, dec_mask))
        x = self.addnorm_2(x, lambda x: self.cross_attn(x, enc_out, enc_out, enc_mask))
        x = self.addnorm_3(x, self.feed_forward)
        return x


class Decoder(nn.Module):
    def __init__(self, d_model, block_list: nn.ModuleList):
        super().__init__()
        self.block_list = block_list
        self.layer_norm = LayerNorm(d_model)

    def forward(self, x, enc_out, enc_mask, dec_mask):
        for block in self.block_list:
            x = block(x, enc_out, enc_mask, dec_mask)
        return self.layer_norm(x)


# ══════════════════════════════════════════════════
# Projection Layer + Full Transformer
# ══════════════════════════════════════════════════

class ProjectionLayer(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        return self.proj(x)


class Transformer(nn.Module):
    def __init__(self, encoder, decoder, src_embed, tgt_embed,
                 src_pos, tgt_pos, projection_layer):
        super().__init__()
        self.encoder          = encoder
        self.decoder          = decoder
        self.src_embed        = src_embed
        self.tgt_embed        = tgt_embed
        self.src_pos          = src_pos
        self.tgt_pos          = tgt_pos
        self.projection_layer = projection_layer

    def encode(self, src, src_mask):
        x = self.src_embed(src)
        x = self.src_pos(x)
        return self.encoder(x, src_mask)

    def decode(self, enc_out, src_mask, tgt, tgt_mask):
        x = self.tgt_embed(tgt)
        x = self.tgt_pos(x)
        return self.decoder(x, enc_out, src_mask, tgt_mask)

    def project(self, x):
        return self.projection_layer(x)


# ══════════════════════════════════════════════════
# Build Model Function
# ══════════════════════════════════════════════════

def build_model(
    src_vocab_size, tgt_vocab_size, src_seq_len, tgt_seq_len,
    d_model=512, num_blocks=6, num_heads=8, dropout_rate=0.1, d_ff=2048
):
    src_embed = EmbeddingLayer(d_model, src_vocab_size)
    tgt_embed = EmbeddingLayer(d_model, tgt_vocab_size)
    src_pos   = PositionalEncoding(d_model, src_seq_len, dropout_rate)
    tgt_pos   = PositionalEncoding(d_model, tgt_seq_len, dropout_rate)

    encoder_blocks = [
        EncoderBlock(
            d_model,
            MultiHeadAttention(d_model, num_heads, dropout_rate),
            FeedForward(d_model, d_ff, dropout_rate),
            dropout_rate
        )
        for _ in range(num_blocks)
    ]

    decoder_blocks = [
        DecoderBlock(
            d_model,
            MultiHeadAttention(d_model, num_heads, dropout_rate),
            MultiHeadAttention(d_model, num_heads, dropout_rate),
            FeedForward(d_model, d_ff, dropout_rate),
            dropout_rate
        )
        for _ in range(num_blocks)
    ]

    model = Transformer(
        Encoder(d_model, nn.ModuleList(encoder_blocks)),
        Decoder(d_model, nn.ModuleList(decoder_blocks)),
        src_embed, tgt_embed, src_pos, tgt_pos,
        ProjectionLayer(d_model, tgt_vocab_size)
    )

    # Xavier initialization for all weight matrices
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)

    return model


# ── Build and move to GPU ──────────────────────────
model = build_model(
    tokenizer_en.get_vocab_size(),
    tokenizer_or.get_vocab_size(),
    max_seq_len, max_seq_len,
    d_model=512
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model built: {total_params:,} parameters")

✅ Model built: 90,213,680 parameters


## STEP 7: Training Loop

**Key improvement:** `gc.collect()` + `torch.cuda.empty_cache()` runs after every epoch.  
This prevents the RAM fragmentation crash that happened at epoch 6 with 700k data.

### How to use:
```python
train_model(start_epoch=0, total_epochs=10)   # First run: epoch 1 → 10
train_model(start_epoch=10, total_epochs=20)  # Resume:    epoch 11 → 20
train_model(start_epoch=20, total_epochs=30)  # Resume:    epoch 21 → 30
```

In [11]:
def train_model(start_epoch=10, total_epochs=20):
    """
    Train OdiaGPT from scratch or resume from checkpoint.

    Args:
        start_epoch : 0 = train from scratch
                      N = resume from ./odiagpt/model_N.pt
        total_epochs: Total epochs to reach (not additional epochs)

    Examples:
        train_model(0, 10)   →  trains epoch 1 to 10
        train_model(10, 20)  →  resumes from epoch 10, trains to epoch 20
        train_model(20, 30)  →  resumes from epoch 20, trains to epoch 30
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, eps=1e-9)

    # Load checkpoint if resuming
    if start_epoch > 0:
        ckpt_path  = f"./odiagpt/model_{start_epoch}.pt"
        checkpoint = torch.load(ckpt_path, map_location=device, weights_only=True)
        model.load_state_dict(checkpoint["model_state_dict"])
        print(f"✅ Resumed from checkpoint: model_{start_epoch}.pt")

    loss_fn = nn.CrossEntropyLoss(
        ignore_index=tokenizer_or.token_to_id("[PAD]"),
        label_smoothing=0.1
    ).to(device)

    print(f"Starting training: epoch {start_epoch+1} → {total_epochs}")
    print(f"Batches per epoch: {len(train_dataloader):,}")
    print("-" * 60)

    for epoch in range(start_epoch + 1, total_epochs + 1):
        model.train()
        epoch_loss = 0
        loop = tqdm(train_dataloader, desc=f"Epoch {epoch:03d}")

        for batch in loop:
            optimizer.zero_grad(set_to_none=True)

            enc_in   = batch["encoder_input"].to(device)
            dec_in   = batch["decoder_input"].to(device)
            enc_mask = batch["encoder_mask"].to(device)
            dec_mask = batch["decoder_mask"].to(device)
            labels   = batch["target_label"].to(device)

            enc_out = model.encode(enc_in, enc_mask)
            dec_out = model.decode(enc_out, enc_mask, dec_in, dec_mask)
            logits  = model.project(dec_out)

            loss = loss_fn(
                logits.view(-1, tokenizer_or.get_vocab_size()),
                labels.view(-1)
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            loop.set_postfix(loss=f"{loss.item():.3f}")

        avg_loss = epoch_loss / len(train_dataloader)
        print(f"\nEpoch {epoch:03d} | Avg Loss: {avg_loss:.4f}")

        # Save checkpoint
        torch.save(
            {"model_state_dict": model.state_dict(), "epoch": epoch},
            f"./odiagpt/model_{epoch}.pt"
        )
        print(f"Saved: model_{epoch}.pt")

        # ══════════════════════════════════════════════════
        # MEMORY CLEANUP — prevents RAM crash on long runs
        # This fixes the epoch 6+ crash with large datasets
        # ══════════════════════════════════════════════════
        gc.collect()
        torch.cuda.empty_cache()

    print("\n" + "=" * 60)
    print("✅ Training complete!")
    print("=" * 60)

In [10]:
# ═══════════════════════════════════════════════════════════
# RUN TRAINING — First time: epoch 0 → 10
# ═══════════════════════════════════════════════════════════
train_model(start_epoch=0, total_epochs=10)

Starting training: epoch 1 → 10
Batches per epoch: 31,250
------------------------------------------------------------


Epoch 001: 100%|██████████| 31250/31250 [4:21:24<00:00,  1.99it/s, loss=6.144]  



Epoch 001 | Avg Loss: 5.7413
Saved: model_1.pt


Epoch 002: 100%|██████████| 31250/31250 [4:20:20<00:00,  2.00it/s, loss=4.581]  



Epoch 002 | Avg Loss: 4.6605
Saved: model_2.pt


...


Epoch 008 | Avg Loss: 3.5277
Saved: model_8.pt


Epoch 009: 100%|██████████| 31250/31250 [11:05:06<00:00,  1.28s/it, loss=3.490]       



Epoch 009 | Avg Loss: 3.4438
Saved: model_9.pt


Epoch 010: 100%|██████████| 31250/31250 [4:28:57<00:00,  1.94it/s, loss=3.605]  



Epoch 010 | Avg Loss: 3.3718
Saved: model_10.pt

✅ Training complete!


In [13]:
train_model(start_epoch=10, total_epochs=20)

✅ Resumed from checkpoint: model_10.pt
Starting training: epoch 11 → 20
Batches per epoch: 31,250
------------------------------------------------------------


Epoch 011: 100%|██████████| 31250/31250 [9:13:19<00:00,  1.06s/it, loss=4.878]  



Epoch 011 | Avg Loss: 5.4833
Saved: model_11.pt


Epoch 012: 100%|██████████| 31250/31250 [10:09:15<00:00,  1.17s/it, loss=4.238] 



Epoch 012 | Avg Loss: 4.4098
Saved: model_12.pt


...


Epoch 017 | Avg Loss: 3.4887
Saved: model_17.pt


Epoch 018: 100%|██████████| 31250/31250 [9:04:14<00:00,  1.04s/it, loss=3.384]  



Epoch 018 | Avg Loss: 3.4059
Saved: model_18.pt


Epoch 019:  41%|████      | 12888/31250 [3:49:21<5:05:57,  1.00it/s, loss=3.366]

: 

In [1]:
import os

for epoch in range(1, 21):
    path = f"./odiagpt/model_{epoch}.pt"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"✅ model_{epoch:02d}.pt  →  {size_mb:.1f} MB")
    else:
        print(f"❌ model_{epoch:02d}.pt  →  NOT FOUND")

✅ model_01.pt  →  344.8 MB
✅ model_02.pt  →  344.8 MB
✅ model_03.pt  →  344.8 MB
✅ model_04.pt  →  344.8 MB
✅ model_05.pt  →  344.8 MB
✅ model_06.pt  →  344.8 MB
✅ model_07.pt  →  344.8 MB
✅ model_08.pt  →  344.8 MB
✅ model_09.pt  →  344.8 MB
✅ model_10.pt  →  344.8 MB
✅ model_11.pt  →  344.8 MB
✅ model_12.pt  →  344.8 MB
✅ model_13.pt  →  344.8 MB
✅ model_14.pt  →  344.8 MB
✅ model_15.pt  →  344.8 MB
✅ model_16.pt  →  344.8 MB
✅ model_17.pt  →  344.8 MB
✅ model_18.pt  →  344.8 MB
✅ model_19.pt  →  344.8 MB
✅ model_20.pt  →  344.8 MB


In [1]:
# ═══════════════════════════════════════════════════════════
# RESUME TRAINING — continue from where you stopped
# Change the numbers to match your last saved epoch
# ═══════════════════════════════════════════════════════════

train_model(start_epoch=10, total_epochs=14)
# train_model(start_epoch=20, total_epochs=30)
# train_model(start_epoch=30, total_epochs=40)

NameError: name 'train_model' is not defined

## STEP 8: Load Model & Translate

In [11]:
# ═══════════════════════════════════════════════════════════
# LOAD CHECKPOINT
# Change BEST_EPOCH to whichever epoch you want to test
# ═══════════════════════════════════════════════════════════
BEST_EPOCH = 10

checkpoint = torch.load(
    f"./odiagpt/model_{BEST_EPOCH}.pt",
    map_location=device,
    weights_only=True
)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()
print(f"✅ Loaded model from epoch {BEST_EPOCH}")

✅ Loaded model from epoch 10


In [1]:
import os
print(os.getcwd())

d:\OdiaLLM


In [12]:
def clean_odia_text(text):
    """Remove special tokens and noise from decoded Odia output"""
    banned = ["[CLS]", "[SEP]", "[PAD]", "[UNK]", "[MASK]",
              "Name", "I / O", "unit", "format", "_u"]
    for token in banned:
        text = text.replace(token, "")
    while "  " in text:
        text = text.replace("  ", " ")
    return text.strip()


def odiagpt_beam(text, beam_size=3, max_len=None):
    """
    Translate English text to Odia using beam search.

    Args:
        text      : English input sentence
        beam_size : 3 is recommended (higher = slower but slightly better)
        max_len   : auto uses training max_seq_len

    Usage:
        odiagpt_beam("The police arrested the accused.")
    """
    if max_len is None:
        max_len = max_seq_len

    model.eval()
    with torch.no_grad():

        # Encode English input
        CLS_EN = tokenizer_en.token_to_id("[CLS]")
        SEP_EN = tokenizer_en.token_to_id("[SEP]")
        PAD_EN = tokenizer_en.token_to_id("[PAD]")

        src = tokenizer_en.encode(text).ids
        src = [CLS_EN] + src[:max_len - 2] + [SEP_EN]
        src += [PAD_EN] * (max_len - len(src))
        src      = torch.tensor(src, dtype=torch.int64).unsqueeze(0).to(device)
        src_mask = (src != PAD_EN).unsqueeze(1).unsqueeze(2).int()
        encoder_output = model.encode(src, src_mask)

        # Beam search init
        CLS = tokenizer_or.token_to_id("[CLS]")
        SEP = tokenizer_or.token_to_id("[SEP]")
        PAD = tokenizer_or.token_to_id("[PAD]")

        beams     = [(torch.tensor([[CLS]], device=device), 0.0)]
        completed = []

        for _ in range(max_len):
            new_beams = []
            for seq, score in beams:
                if seq[0, -1].item() == SEP:
                    completed.append((seq, score))
                    continue
                dec_mask  = causal_mask(seq.size(1)).to(device)
                out       = model.decode(encoder_output, src_mask, seq, dec_mask)
                logits    = model.project(out[:, -1])
                log_probs = torch.log_softmax(logits, dim=-1)
                topk_probs, topk_ids = torch.topk(log_probs, beam_size)
                for i in range(beam_size):
                    next_id    = topk_ids[0, i].item()
                    next_score = score + topk_probs[0, i].item()
                    new_seq    = torch.cat(
                        [seq, torch.tensor([[next_id]], device=device)], dim=1
                    )
                    new_beams.append((new_seq, next_score))

            beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]
            if len(completed) >= beam_size:
                break

        best_seq = completed[0][0] if completed else beams[0][0]
        tokens   = [t for t in best_seq[0].tolist() if t not in {CLS, SEP, PAD}]
        decoded  = tokenizer_or.decode(tokens)
        return clean_odia_text(decoded)


print("✅ Beam search ready!")
print("Usage: odiagpt_beam('Your English sentence here')")

✅ Beam search ready!
Usage: odiagpt_beam('Your English sentence here')


In [13]:
# ═══════════════════════════════════════════════════════════
# QUICK TRANSLATION TEST
# ═══════════════════════════════════════════════════════════
test_sentences = [
    "The government announced a new policy.",
    "The police arrested the accused.",
    "Heavy rain fell in Odisha.",
    "The court gave its verdict today.",
    "India won the cricket match.",
    "The minister resigned from his post.",
    "The train was delayed due to heavy fog.",
    "Farmers received compensation for crop damage.",
]

print("QUICK TRANSLATION TEST")
print("=" * 65)
for sent in test_sentences:
    translation = odiagpt_beam(sent, beam_size=3)
    print(f"EN: {sent}")
    print(f"OR: {translation}")
    print("-" * 65)

QUICK TRANSLATION TEST
EN: The government announced a new policy.
OR: ଏନେଇ ସରକାର ନୂଆ ଯୋଜନା ଘୋଷଣା କରିଛନ୍ତି ।
-----------------------------------------------------------------
EN: The police arrested the accused.
OR: ପୋଲିସ ଅଭିଯୁକ୍ତକୁ ଗିରଫ କରିଛି ।
-----------------------------------------------------------------
EN: Heavy rain fell in Odisha.
OR: ଲଘୁଚାପ ପ୍ରଭାବରେ ଓଡ଼ିଶାରେ ବର୍ଷା ହୋଇଛି ।
-----------------------------------------------------------------
EN: The court gave its verdict today.
OR: କୋର୍ଟ ଆଜି ରାୟ ଶୁଣାଇଛନ୍ତି ।
-----------------------------------------------------------------
EN: India won the cricket match.
OR: ଏହି ମ୍ୟାଚରେ ଭାରତ ବିଜୟୀ ହୋଇଥିଲା ।
-----------------------------------------------------------------
EN: The minister resigned from his post.
OR: ମନ୍ତ୍ରୀ ପଦରୁ ଇସ୍ତଫା ଦେଇଥିଲେ ।
-----------------------------------------------------------------
EN: The train was delayed due to heavy fog.
OR: ପ୍ରବଳ କୁହୁଡ଼ି ଯୋଗୁଁ ଟ୍ରେନ ଚଳାଚଳରେ ବିଳମ୍ବ ହୋଇଛି ।
------------------------------------------

In [14]:
# ═══════════════════════════════════════════════════════════
# TRANSLATE YOUR OWN SENTENCE
# Change the sentence below and run
# ═══════════════════════════════════════════════════════════
my_sentence = "The hospital was inaugurated by the Chief Minister."
result = odiagpt_beam(my_sentence, beam_size=3)
print(f"EN: {my_sentence}")
print(f"OR: {result}")

EN: The hospital was inaugurated by the Chief Minister.
OR: ମୁଖ୍ୟମନ୍ତ୍ରୀ ଏହାକୁ ଉଦଘାଟନ କରିଥିଲେ ।


## STEP 9: Evaluation

Three evaluation methods:
1. **Manual test** — 10 fixed sentences with references, shows chrF per sentence
2. **Unseen validation** — 20 random sentences from validation set (honest score)
3. **Single sentence scorer** — test any sentence with reference

In [15]:
# ═══════════════════════════════════════════════════════════
# EVALUATION 1: Manual test with chrF per sentence
# ═══════════════════════════════════════════════════════════
test_cases = [
    ("The police arrested two people.",         "ପୋଲିସ ଦୁଇ ଜଣଙ୍କୁ ଗିରଫ କଲା ।"),
    ("Heavy rain fell in Odisha.",              "ଓଡ଼ିଶାରେ ପ୍ରବଳ ବର୍ଷା ହୋଇଛି ।"),
    ("The court gave its verdict today.",       "କୋର୍ଟ ଆଜି ରାୟ ଦେଇଛି ।"),
    ("The government announced a new scheme.",  "ସରକାର ଏକ ନୂଆ ଯୋଜନା ଘୋଷଣା କଲେ ।"),
    ("Two people were killed in the accident.", "ଦୁର୍ଘଟଣାରେ ଦୁଇ ଜଣ ମୃତ୍ୟୁ ବରଣ କଲେ ।"),
    ("The election results were declared.",     "ନିର୍ବାଚନ ଫଳ ଘୋଷଣା ହୋଇଛି ।"),
    ("India won the cricket match.",            "ଭାରତ କ୍ରିକେଟ ମ୍ୟାଚ ଜିତିଛି ।"),
    ("The school was closed due to rain.",      "ବର୍ଷା କାରଣରୁ ବିଦ୍ୟାଳୟ ବନ୍ଦ ହୋଇଗଲା ।"),
    ("The minister resigned from his post.",    "ମନ୍ତ୍ରୀ ନିଜ ପଦରୁ ଇସ୍ତଫା ଦେଲେ ।"),
    ("Farmers protested on the highway.",       "କୃଷକମାନେ ରାଜପଥରେ ଧରଣା ଦେଲେ ।"),
]

predictions = []
references  = []

print(f"{'#':<3} {'ENGLISH':<45} {'chrF'}")
print("=" * 70)

for i, (en, ref) in enumerate(test_cases):
    pred  = odiagpt_beam(en, beam_size=3)
    score = sacrebleu.sentence_chrf(pred, [ref]).score
    predictions.append(pred)
    references.append(ref)
    print(f"{i+1:<3} {en:<45} {score:.1f}")

print("=" * 70)
chrf_overall = sacrebleu.corpus_chrf(predictions, [references])
bleu_overall = sacrebleu.corpus_bleu(predictions, [references], tokenize="char")
avg_chrf     = sum(sacrebleu.sentence_chrf(p, [r]).score for p, r in zip(predictions, references)) / len(predictions)

print(f"\nOverall chrF     : {chrf_overall.score:.2f}")
print(f"Overall BLEU     : {bleu_overall.score:.2f}")
print(f"Avg sentence chrF: {avg_chrf:.2f}")

print("\nInterpretation:")
print("  chrF < 30   → keep training")
print("  chrF 30-45  → decent for low-resource from scratch")
print("  chrF 45-55  → good")
print("  chrF 55+    → excellent for Odia from scratch")

print("\n--- Predictions vs References ---")
for i, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"\n[{i+1}] REF : {ref}")
    print(f"     PRED: {pred}")

#   ENGLISH                                       chrF
1   The police arrested two people.               74.2
2   Heavy rain fell in Odisha.                    52.1
3   The court gave its verdict today.             60.0
4   The government announced a new scheme.        63.2
5   Two people were killed in the accident.       62.3
6   The election results were declared.           48.7
7   India won the cricket match.                  23.4
8   The school was closed due to rain.            20.2
9   The minister resigned from his post.          66.2
10  Farmers protested on the highway.             25.3

Overall chrF     : 48.42
Overall BLEU     : 48.71
Avg sentence chrF: 49.55

Interpretation:
  chrF < 30   → keep training
  chrF 30-45  → decent for low-resource from scratch
  chrF 45-55  → good
  chrF 55+    → excellent for Odia from scratch

--- Predictions vs References ---

[1] REF : ପୋଲିସ ଦୁଇ ଜଣଙ୍କୁ ଗିରଫ କଲା ।
     PRED: ପୁଲିସ ଦୁଇ ଜଣଙ୍କୁ ଗିରଫ କରିଛି ।

[2] REF : ଓଡ଼ିଶାରେ ପ୍ରବଳ ବର୍ଷା ହୋଇ

In [16]:
# ═══════════════════════════════════════════════════════════
# EVALUATION 2: Real unseen validation sentences
# This is the HONEST score — model never saw these sentences
# ═══════════════════════════════════════════════════════════
random.seed(42)
val_samples = random.sample(list(raw_val_dataset), 20)

unseen_preds = []
unseen_refs  = []

print("UNSEEN VALIDATION SENTENCES")
print("(These were never seen during training — this is the real accuracy)\n")

for sample in val_samples:
    en   = sample["src"]
    ref  = sample["tgt"]
    pred = odiagpt_beam(en, beam_size=3)
    score = sacrebleu.sentence_chrf(pred, [ref]).score
    unseen_preds.append(pred)
    unseen_refs.append(ref)
    print(f"EN  : {en}")
    print(f"REF : {ref}")
    print(f"PRED: {pred}")
    print(f"chrF: {score:.1f}")
    print("-" * 60)

chrf_unseen = sacrebleu.corpus_chrf(unseen_preds, [unseen_refs])
print(f"\n{'='*60}")
print(f"Real chrF on unseen data: {chrf_unseen.score:.2f}")
print(f"{'='*60}")

UNSEEN VALIDATION SENTENCES
(These were never seen during training — this is the real accuracy)

EN  : Once it is done, work will start.
REF : ଥରେ ଏହା ପ୍ରସ୍ତୁତ ହେଲା ପରେ ତୁରନ୍ତ କାମ ଆରମ୍ଭ କରାଯିବ ।
PRED: ଏହା ଥରେ ହେଲେ କାମ ଆରମ୍ଭ ହେବ ।
chrF: 25.1
------------------------------------------------------------
EN  : It has many advantages.
REF : ଏହାର ଅନେକ ଫାଇଦା ଅଛି।
PRED: ଏହାର ଅନେକ ଫାଇଦା ରହିଛି ।
chrF: 76.1
------------------------------------------------------------
EN  : And Simon Peter followed Jesus, and so did another disciple: that disciple was known unto the high priest, and went in with Jesus into the palace of the high priest.
REF : ଶିମାନେ ପିତର ଓ ଅନ୍ୟ ଜଣେ ଶିଷ୍ଯ ଯୀଶୁଙ୍କ ସହିତ ଗଲେ। ସହେି ଶିଷ୍ଯଜଣ ମହାୟାଜକଙ୍କୁ ଜାଣିଥିଲେ। ତେଣୁ ସେ ଯୀଶୁଙ୍କ ସହିତ ମହାୟାଜକଙ୍କ ଘର ଅଗଣା ପର୍ୟ୍ଯନ୍ତ ଗଲେ।
PRED: ତା ' ପରେ ପିତର , ପିତର , ଯାକୁବ , ପିତର , ଯୋହନ ଅନ୍ୟ ଜଣେ ଶିଷ୍ଯ ଥିଲେ । ସମାନେେ ଯୀଶୁଙ୍କ ଆଗ ରେ ମହାୟାଜକ ଥିଲେ । ସମାନେେ ଯୀଶୁଙ୍କ ସହିତ ସ୍ବର୍ଗ ୀୟ ମହାୟାଜକ ଥିଲେ ।
chrF: 41.3
------------------------------------------------------------

In [17]:
# ═══════════════════════════════════════════════════════════
# EVALUATION 3: Single sentence scorer
# Use this to test any English sentence with a reference
# ═══════════════════════════════════════════════════════════
def test_single(english, reference_odia=""):
    """
    Translate one sentence and optionally score against reference.

    Usage:
        test_single("The police arrested two people.", "ପୋଲିସ ଦୁଇ ଜଣଙ୍କୁ ଗିରଫ କଲା ।")
        test_single("The police arrested two people.")  # no reference, just translates
    """
    pred = odiagpt_beam(english, beam_size=3)
    print("=" * 60)
    print(f"EN  : {english}")
    if reference_odia:
        score = sacrebleu.sentence_chrf(pred, [reference_odia]).score
        print(f"REF : {reference_odia}")
        print(f"PRED: {pred}")
        print(f"chrF: {score:.2f}")
    else:
        print(f"OR  : {pred}")
    print("=" * 60)
    return pred


# ── Test examples — change these to your own sentences ──────
test_single(
    "The election results were announced today.",
    "ଆଜି ନିର୍ବାଚନ ଫଳ ଘୋଷଣା କରାଗଲା ।"
)

test_single(
    "Farmers are demanding better prices for their crops.",
    "କୃଷକମାନେ ସେମାନଙ୍କ ଫସଲ ପାଇଁ ଭଲ ମୂଲ୍ୟ ଦାବି କରୁଛନ୍ତି ।"
)

test_single(
    "The hospital was inaugurated by the Chief Minister.",
    "ମୁଖ୍ୟମନ୍ତ୍ରୀ ଡାକ୍ତରଖାନାର ଉଦ୍ଘାଟନ କଲେ ।"
)

# Translate without reference
test_single("The train was delayed due to heavy fog.")

EN  : The election results were announced today.
REF : ଆଜି ନିର୍ବାଚନ ଫଳ ଘୋଷଣା କରାଗଲା ।
PRED: ନିର୍ବାଚନ ଫଳାଫଳ ଘୋଷଣା ହେଲା ।
chrF: 56.22
EN  : Farmers are demanding better prices for their crops.
REF : କୃଷକମାନେ ସେମାନଙ୍କ ଫସଲ ପାଇଁ ଭଲ ମୂଲ୍ୟ ଦାବି କରୁଛନ୍ତି ।
PRED: ଚାଷୀ ମାନେ ଫସଲର ଉଚିତ୍ ମୂଲ୍ୟ ମିଳୁ ବୋଲି ଦାବି କରିଛନ୍ତି ।
chrF: 31.62
EN  : The hospital was inaugurated by the Chief Minister.
REF : ମୁଖ୍ୟମନ୍ତ୍ରୀ ଡାକ୍ତରଖାନାର ଉଦ୍ଘାଟନ କଲେ ।
PRED: ମୁଖ୍ୟମନ୍ତ୍ରୀ ଏହାକୁ ଉଦଘାଟନ କରିଥିଲେ ।
chrF: 44.92
EN  : The train was delayed due to heavy fog.
OR  : ପ୍ରବଳ କୁହୁଡ଼ି ଯୋଗୁଁ ଟ୍ରେନ ଚଳାଚଳରେ ବିଳମ୍ବ ହୋଇଛି ।


'ପ୍ରବଳ କୁହୁଡ଼ି ଯୋଗୁଁ ଟ୍ରେନ ଚଳାଚଳରେ ବିଳମ୍ବ ହୋଇଛି ।'